# Cambridge Bay -- Cross-Region Generalization: Sentinel-1 Patch Extraction

Cross-region generalization test requested by Michel: does a model
**trained only on Tuktoyaktuk** produce useful roughness predictions on a
region it has never seen? Cambridge Bay's LiDAR ground truth already
exists (provided directly, `input_data/lidar_patches_cambridge_extracted/
lidar_patches_cambridge/`, 2112 patches, verified genuinely Cambridge Bay
via CRS reprojection -- 33.4km from the townsite, correct UTM zone 13N)
-- so this notebook only needs to do the Sentinel-1 half: match PC-RTC
Sentinel-1 imagery to these *already-existing* LiDAR patches, exactly the
way `02_patch_extraction.ipynb` did for Tuktoyaktuk.

**No training happens on this data at all.** Every patch extracted here
is held out entirely for inference with an already-trained checkpoint
(`09`'s or `10`'s) in a follow-up notebook -- that's what makes this a
genuine generalization test rather than another in-region result.

**Identical logic to `02_patch_extraction.ipynb`** -- only the
region-specific config values differ (`LIDAR_DIR`, `OUT_S1_DIR`, search
date). The AOI-building, windowed VV+VH merge, and
`build_s1_products_from_corrected`/`extract_lidar_matched_s1_patches`
matching functions are copied verbatim, unmodified, so any result
difference from Tuktoyaktuk is attributable to the region itself, not a
different extraction method. The search date, `2024-04-18`, was already
present in `02`'s own `DATE_BY_REGION` dict (`{'cambridge':
dt.date(2024, 4, 18)}`), matching the `CBApr18` LiDAR survey date.

## Setup

In [51]:
import os
import json
import datetime as dt
import glob as glob_module
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import rasterio
from rasterio.warp import transform_bounds, transform_geom
from rasterio.windows import Window, from_bounds
from shapely.geometry import box, shape
from shapely.ops import unary_union

import pystac_client
import planetary_computer
from dotenv import load_dotenv

load_dotenv()
if os.environ.get('PC_SDK_SUBSCRIPTION_KEY'):
    planetary_computer.settings.set_subscription_key(os.environ['PC_SDK_SUBSCRIPTION_KEY'])

## Configuration

Only these values differ from `02_patch_extraction.ipynb`'s Tuktoyaktuk
run: `REGION`, `LIDAR_DIR` (Cambridge Bay's already-provided patches,
note the nested path from how the zip extracted), and `OUT_S1_DIR`. The
search date (`2024-04-18`) comes from the same `DATE_BY_REGION` dict `02`
already had defined.

In [52]:
# Region-specific paths and output directories -- SEARCH_DAYS removed, cell 6 now does an unrestricted search instead of a fixed window
REPO_DIR = Path('/cs/student/project_msc/2025/aibh/jiayiche')
INPUT_DIR = REPO_DIR / 'input_data'
REGION = 'cambridge'
LIDAR_DIR = INPUT_DIR / 'lidar_patches_cambridge_extracted' / 'lidar_patches_cambridge'
DATE_BY_REGION = {'pondinlet': dt.date(2024, 4, 26), 'cambridge': dt.date(2024, 4, 18), 'tuk': dt.date(2024, 4, 16)}

PATCH_SIZE = 256
S1_PATCH_SIZE = int(round(PATCH_SIZE / 10))  # 26 px @ 10m

MERGED_DIR = REPO_DIR / 'raw_data' / f'{REGION}_pc_rtc_merged'
OUT_S1_DIR = INPUT_DIR / f's1_patches_{REGION}_pcrtc'
MERGED_DIR.mkdir(parents=True, exist_ok=True)
OUT_S1_DIR.mkdir(parents=True, exist_ok=True)
print('LIDAR_DIR:', LIDAR_DIR)
print('MERGED_DIR:', MERGED_DIR)
print('OUT_S1_DIR:', OUT_S1_DIR)

LIDAR_DIR: /cs/student/project_msc/2025/aibh/jiayiche/input_data/lidar_patches_cambridge_extracted/lidar_patches_cambridge
MERGED_DIR: /cs/student/project_msc/2025/aibh/jiayiche/raw_data/cambridge_pc_rtc_merged
OUT_S1_DIR: /cs/student/project_msc/2025/aibh/jiayiche/input_data/s1_patches_cambridge_pcrtc


## 1. Build the AOI and select the nearest available dual-pol scenes

A fixed `SEARCH_DAYS` window around the survey date found **zero**
scenes -- Planetary Computer's `sentinel-1-rtc` archive for this AOI has
a complete 3-year gap (2022-05 to 2025-04) that swallows
`2024-04-18` entirely (documented in `CONCEPTS.md`). This cell instead
does an **unrestricted** search, explicitly filters to scenes that have
**both** `vv` and `vh` assets (learned the hard way on the Pond Inlet
notebook -- not every scene in this collection is dual-pol; some AOIs
are covered by HH-only or HH+HV acquisitions instead), and picks the
`N_SCENES` closest to the survey date among the dual-pol ones only.

The nearest available dual-pol scenes turn out to be from **May 2025**,
~402 days after the survey and a different season (post-thaw summer vs.
the April/frozen-season survey) -- a real, single confound, accepted
here in preference to Pond Inlet's three-way confound (region + date +
polarization mode).

In [53]:
# Build the Cambridge Bay AOI, then search all time and keep only dual-pol (VV+VH) scenes nearest the survey date
def aoi_from_lidar_patches(patches_dir, max_files=300, workers=8):
    paths = sorted(patches_dir.glob('lidar_patch_*.tif'))
    if not paths:
        raise FileNotFoundError(f'No LiDAR patches found in {patches_dir}')
    if len(paths) > max_files:
        stride = len(paths) / max_files
        paths = [paths[int(i * stride)] for i in range(max_files)]

    def read_bounds(path):
        with rasterio.open(path) as src:
            return src.crs, src.bounds

    results = []
    with ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(read_bounds, p) for p in paths]
        for future in as_completed(futures):
            results.append(future.result())
    crs = results[0][0]
    native = unary_union([box(*bounds) for _, bounds in results])
    geojson = transform_geom(crs, 'EPSG:4326', native.__geo_interface__)
    return shape(geojson).buffer(0)

aoi = aoi_from_lidar_patches(LIDAR_DIR)
aoi_ll = aoi.convex_hull
print('AOI bounds (lon, lat):', aoi_ll.bounds)

catalog = pystac_client.Client.open(
    'https://planetarycomputer.microsoft.com/api/stac/v1',
    modifier=planetary_computer.sign_inplace,
)
search = catalog.search(collections=['sentinel-1-rtc'], intersects=aoi_ll.__geo_interface__)
all_items = list(search.items())
print(f'Total scenes for this AOI (all time): {len(all_items)}')

dualpol_items = [it for it in all_items if 'vv' in it.assets and 'vh' in it.assets]
print(f'{len(dualpol_items)} / {len(all_items)} scenes have both VV and VH')

from collections import Counter
pol_counts = Counter(tuple(sorted(k for k in it.assets if k in ('vv', 'vh', 'hh', 'hv'))) for it in all_items)
print('Polarization mix across all scenes:', dict(pol_counts))

N_SCENES = 3  # matches CONTEXT_K used throughout this project
target = dt.datetime.combine(DATE_BY_REGION[REGION], dt.time(), tzinfo=dt.timezone.utc)
items = sorted(dualpol_items, key=lambda it: abs((it.datetime - target).total_seconds()))[:N_SCENES]
items = sorted(items, key=lambda it: it.datetime)  # chronological order for t0/t1/t2 naming

print(f'\nSelected {len(items)} nearest dual-pol scenes to {DATE_BY_REGION[REGION].isoformat()}:')
for i, item in enumerate(items):
    days_off = (item.datetime - target).total_seconds() / 86400
    print(f'  t{i}: {item.id} | {item.datetime} | {days_off:+.1f} days from survey')

AOI bounds (lon, lat): (-105.9884493732042, 68.95816560420016, -105.67918440507977, 69.02930644230831)
Total scenes for this AOI (all time): 301
194 / 301 scenes have both VV and VH
Polarization mix across all scenes: {('vh', 'vv'): 194, ('hh', 'hv'): 107}

Selected 3 nearest dual-pol scenes to 2024-04-18:
  t0: S1C_IW_GRDH_1SDV_20250525T003630_20250525T003655_002480_rtc | 2025-05-25 00:36:42.724974+00:00 | +402.0 days from survey
  t1: S1C_IW_GRDH_1SDV_20250527T002011_20250527T002036_002509_rtc | 2025-05-27 00:20:24.001200+00:00 | +404.0 days from survey
  t2: S1C_IW_GRDH_1SDV_20250601T002820_20250601T002851_002582_rtc | 2025-06-01 00:28:36.074235+00:00 | +409.0 days from survey


## 2. Merge VV+VH into local 2-band GeoTIFFs, reprojected to a consistent grid

Same windowed-HTTPS-read approach as before, but with one addition: each
scene gets **reprojected into the LiDAR patches' own CRS at a fixed 10m
grid** before being written to disk. This is necessary because
Sentinel-1 scenes are assigned a UTM zone based on the *whole scene's*
footprint center, not this AOI specifically -- `t0` (2025-05-25) turned
out to be delivered in zone 12N while the LiDAR patches (and `t1`/`t2`)
are in zone 13N. Without reprojecting first, windowing `t0` against the
LiDAR patches' native-zone bounds computed a distorted 28x28 window
instead of the expected 26x26, causing every single patch to fail the
matcher's exact-size check (confirmed by tracing one patch through the
matching logic step by step). Reprojecting every timestep to the same
target grid up front makes the matching step insensitive to whichever
native zone Planetary Computer happened to deliver each scene in.

In [54]:
# Merge VV+VH per scene, reprojecting each into the LiDAR patches' CRS at 10m so all timesteps share identical grid geometry
from rasterio.warp import calculate_default_transform, reproject, Resampling

lidar_sample_path = sorted(LIDAR_DIR.glob('lidar_patch_*.tif'))[0]
with rasterio.open(lidar_sample_path) as s:
    TARGET_CRS = s.crs
print('Target CRS (from LiDAR patches):', TARGET_CRS)

merged_paths = []
merged_attrs = []

for i, item in enumerate(items):
    with rasterio.open(item.assets['vv'].href) as vv_src:
        aoi_bounds_scene_crs = transform_bounds('EPSG:4326', vv_src.crs, *aoi_ll.bounds)
        window = from_bounds(*aoi_bounds_scene_crs, transform=vv_src.transform)
        vv = vv_src.read(1, window=window)
        src_transform = rasterio.windows.transform(window, vv_src.transform)
        src_crs = vv_src.crs
    with rasterio.open(item.assets['vh'].href) as vh_src:
        vh_window = from_bounds(*transform_bounds('EPSG:4326', vh_src.crs, *aoi_ll.bounds), transform=vh_src.transform)
        vh = vh_src.read(1, window=vh_window)

    h = min(vv.shape[0], vh.shape[0])
    w = min(vv.shape[1], vh.shape[1])
    stacked = np.stack([vv[:h, :w], vh[:h, :w]]).astype(np.float32)

    src_bounds = rasterio.transform.array_bounds(h, w, src_transform)
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src_crs, TARGET_CRS, w, h, *src_bounds, resolution=(10.0, 10.0)
    )
    # fill with NaN, not 0 -- any destination pixel the reprojection doesn't actually cover
    # (e.g. a border strip left over from warping between UTM zones) must read as invalid,
    # not as a fake all-zero SAR return, so the matcher's existing NaN check catches it
    dst = np.full((2, dst_height, dst_width), np.nan, dtype=np.float32)
    reproject(
        source=stacked, destination=dst,
        src_transform=src_transform, src_crs=src_crs,
        dst_transform=dst_transform, dst_crs=TARGET_CRS,
        src_nodata=np.nan, dst_nodata=np.nan,
        resampling=Resampling.bilinear,
    )

    out_path = MERGED_DIR / f't{i}.tif'
    meta = {'driver': 'GTiff', 'count': 2, 'height': dst_height, 'width': dst_width,
            'dtype': 'float32', 'crs': TARGET_CRS, 'transform': dst_transform, 'nodata': np.nan}
    with rasterio.open(out_path, 'w', **meta) as dst_f:
        dst_f.write(dst)
    merged_paths.append(str(out_path))

    props = item.properties
    merged_attrs.append({
        'acquisition_date': item.datetime.date().isoformat(),
        'orbit_direction': 'ASCENDING' if props.get('sat:orbit_state') == 'ascending' else 'DESCENDING',
        'relative_orbit_number': props.get('sat:relative_orbit'),
    })
    print(f'Wrote t{i}.tif: shape={dst.shape}, crs={TARGET_CRS}, nan_frac={float(np.isnan(dst).mean()):.4f}, zero_frac={float((dst == 0.0).mean()):.4f}')

attrs_json_path = MERGED_DIR / 'attrs.json'
with open(attrs_json_path, 'w') as jf:
    json.dump(merged_attrs, jf, indent=2)
print('Wrote', attrs_json_path)

Target CRS (from LiDAR patches): EPSG:32613
Wrote t0.tif: shape=(2, 1018, 1381), crs=EPSG:32613, nan_frac=0.1732, zero_frac=0.0000
Wrote t1.tif: shape=(2, 810, 1248), crs=EPSG:32613, nan_frac=0.0000, zero_frac=0.0000
Wrote t2.tif: shape=(2, 810, 1248), crs=EPSG:32613, nan_frac=0.0000, zero_frac=0.0000
Wrote /cs/student/project_msc/2025/aibh/jiayiche/raw_data/cambridge_pc_rtc_merged/attrs.json


**Check before continuing**: confirm `nodata_frac` is near 0 for every
`t{i}.tif` above. If any date shows a high nodata fraction, that scene's
footprint likely doesn't fully cover the Cambridge Bay AOI -- worth
excluding it rather than propagating a partially-empty product into the
patch matcher below.

## 3. Match against Cambridge Bay's existing LiDAR patches

Same `build_s1_products_from_corrected`/`extract_lidar_matched_s1_patches`
functions as `02`, copied verbatim -- only `LIDAR_DIR` differs.

In [55]:
def build_s1_products_from_corrected(geotiff_paths, attrs_jsons=None):
    products = []
    for i, path in enumerate(geotiff_paths):
        src = rasterio.open(path)
        attrs = attrs_jsons[i] if attrs_jsons and i < len(attrs_jsons) else None
        products.append({"src": src, "crs": src.crs, "transform": src.transform,
                          "height": src.height, "width": src.width, "attrs": attrs})
    if not products:
        raise ValueError("No Sentinel-1 products loaded -- check geotiff_paths.")
    return products


def close_products(products):
    for p in products:
        p["src"].close()


def extract_lidar_matched_s1_patches(lidar_patches_dir, sentinel1_products, s1_patch_size,
                                      out_s1_dir, pattern="lidar_patch_*.tif", max_nan_frac=0.02):
    lidar_paths = sorted(glob_module.glob(os.path.join(str(lidar_patches_dir), pattern)))
    print(f"Found {len(lidar_paths)} existing LiDAR patches to match against "
          f"{len(sentinel1_products)} Sentinel-1 product(s).")

    n_written, n_skipped, n_skipped_nan = 0, 0, 0

    for idx, lp in enumerate(lidar_paths):
        if idx % 100 == 0:
            print(f"  ...processed {idx}/{len(lidar_paths)} "
                  f"(written: {n_written}, skipped: {n_skipped}, skipped-NaN: {n_skipped_nan})")

        patch_id = os.path.splitext(os.path.basename(lp))[0].split("_")[-1]

        with rasterio.open(lp) as lsrc:
            lidar_bounds = lsrc.bounds
            lidar_crs = lsrc.crs

        s1_patches, s1_transforms = [], []
        ok = True
        has_too_much_nan = False
        for prod in sentinel1_products:
            try:
                s1_bounds = transform_bounds(lidar_crs, prod["crs"], *lidar_bounds, densify_pts=21)
                window = from_bounds(*s1_bounds, transform=prod["transform"]).round_offsets().round_lengths()
                r0, c0 = int(window.row_off), int(window.col_off)
                hh, ww = int(window.height), int(window.width)

                if (hh, ww) != (s1_patch_size, s1_patch_size):
                    ok = False; break
                if r0 < 0 or c0 < 0:
                    ok = False; break
                if r0 + s1_patch_size > prod["height"] or c0 + s1_patch_size > prod["width"]:
                    ok = False; break

                read_window = Window(c0, r0, s1_patch_size, s1_patch_size)
                patch = prod["src"].read(window=read_window)
                if patch.shape[1:] != (s1_patch_size, s1_patch_size):
                    ok = False; break

                nan_frac = float(np.mean(np.isnan(patch)))
                if nan_frac > max_nan_frac:
                    has_too_much_nan = True
                    break

                s1_patches.append(patch)
                s1_transforms.append(rasterio.windows.transform(read_window, prod["transform"]))
            except Exception:
                ok = False
                break

        if has_too_much_nan:
            n_skipped_nan += 1
            continue

        if not ok or len(s1_patches) != len(sentinel1_products):
            n_skipped += 1
            continue

        patch_dir = os.path.join(str(out_s1_dir), f"s1_patch_{patch_id}")
        os.makedirs(patch_dir, exist_ok=True)

        attrs_list = []
        for ti, (prod, patch, tr) in enumerate(zip(sentinel1_products, s1_patches, s1_transforms)):
            meta = {
                "driver": "GTiff", "count": patch.shape[0],
                "height": s1_patch_size, "width": s1_patch_size,
                "dtype": "float32", "crs": prod["crs"], "transform": tr,
            }
            with rasterio.open(os.path.join(patch_dir, f"t{ti}.tif"), "w", **meta) as dst:
                dst.write(patch.astype(np.float32))
            attrs_list.append(prod.get("attrs"))

        with open(os.path.join(patch_dir, "attrs.json"), "w") as jf:
            json.dump(attrs_list, jf, indent=2)

        n_written += 1

    print(f"Done. Matched: {n_written}, skipped (out of bounds/wrong size): {n_skipped}, "
          f"skipped (too much NaN, >{max_nan_frac:.0%}): {n_skipped_nan}")
    return n_written, n_skipped

In [56]:
products_pcrtc = build_s1_products_from_corrected(merged_paths, attrs_jsons=merged_attrs)
extract_lidar_matched_s1_patches(LIDAR_DIR, products_pcrtc, S1_PATCH_SIZE, OUT_S1_DIR)
close_products(products_pcrtc)

Found 2112 existing LiDAR patches to match against 3 Sentinel-1 product(s).
  ...processed 0/2112 (written: 0, skipped: 0, skipped-NaN: 0)
  ...processed 100/2112 (written: 100, skipped: 0, skipped-NaN: 0)
  ...processed 200/2112 (written: 200, skipped: 0, skipped-NaN: 0)
  ...processed 300/2112 (written: 300, skipped: 0, skipped-NaN: 0)
  ...processed 400/2112 (written: 400, skipped: 0, skipped-NaN: 0)
  ...processed 500/2112 (written: 500, skipped: 0, skipped-NaN: 0)
  ...processed 600/2112 (written: 600, skipped: 0, skipped-NaN: 0)
  ...processed 700/2112 (written: 700, skipped: 0, skipped-NaN: 0)
  ...processed 800/2112 (written: 800, skipped: 0, skipped-NaN: 0)
  ...processed 900/2112 (written: 900, skipped: 0, skipped-NaN: 0)
  ...processed 1000/2112 (written: 1000, skipped: 0, skipped-NaN: 0)
  ...processed 1100/2112 (written: 1100, skipped: 0, skipped-NaN: 0)
  ...processed 1200/2112 (written: 1200, skipped: 0, skipped-NaN: 0)
  ...processed 1300/2112 (written: 1300, skipped: 0

## 4. Verify the output

Same checks used throughout this project: expected fixed window size (no
CRS mismatch), and a per-band sanity check that patches aren't full of
NaN/zero placeholder values.

In [58]:
sample_patches = sorted(glob_module.glob(os.path.join(str(OUT_S1_DIR), 's1_patch_*')))
lidar_total = len(glob_module.glob(os.path.join(str(LIDAR_DIR), 'lidar_patch_*.tif')))
print(f'Total patches written: {len(sample_patches)} (out of {lidar_total} Cambridge Bay LiDAR patches)')

if sample_patches:
    with rasterio.open(os.path.join(sample_patches[0], 't0.tif')) as src:
        print('Sample patch shape:', src.shape, f'(expect {S1_PATCH_SIZE}x{S1_PATCH_SIZE})')
    for f in sorted(glob_module.glob(os.path.join(sample_patches[0], 't*.tif'))):
        with rasterio.open(f) as src:
            arr = src.read()
            print(f'  {os.path.basename(f)}: finite_frac={np.isfinite(arr).mean():.4f}, '
                  f'nonzero_frac={(arr != 0).mean():.4f}, min/max={arr.min():.5f}/{arr.max():.5f}')

from collections import Counter
counts = Counter(len(glob_module.glob(os.path.join(p, 't*.tif'))) for p in sample_patches)
print('Distribution of timesteps per patch:', dict(sorted(counts.items())))

Total patches written: 2101 (out of 2112 Cambridge Bay LiDAR patches)
Sample patch shape: (26, 26) (expect 26x26)
  t0.tif: finite_frac=1.0000, nonzero_frac=1.0000, min/max=0.00100/0.12334
  t1.tif: finite_frac=1.0000, nonzero_frac=1.0000, min/max=0.00125/0.12852
  t2.tif: finite_frac=1.0000, nonzero_frac=1.0000, min/max=0.00044/0.12768
Distribution of timesteps per patch: {3: 2101}


## Next step (separate notebook)

If the checks above look healthy (patch size correct, high
finite/nonzero fractions, no CRS-mismatch skips), the next step is a
pure-inference notebook: load `09`'s (or `10`'s) already-trained
checkpoint, run it on these Cambridge Bay patches with **no
retraining and no train/val split** (every patch is held-out test data),
and compute the same reconstruction metrics used throughout this project
for direct comparison against the Tuktoyaktuk in-region validation
numbers. `CONTEXT_K` may need reducing from `3` if fewer than 3 usable
Sentinel-1 products survive the matching step above -- check the
"Distribution of timesteps per patch" printout to confirm.